# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Teninlanimi/flyrank-interenship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1

Paper finding:
Growing content had a different structural profile from declining content: growing pages were longer and younger on average.

Methodology question:
How exactly is the growth/decline outcome defined, and does the label represent a meaningful outcome for prediction rather than simply a categorization derived from the same performance data being analyzed?

Why this matters:
The paper defines trend direction from the change in impressions between two 30-day periods. Because this is an observational comparison, relationships between the trend label and other performance variables should be interpreted as descriptive rather than causal.

Finding 4

Paper finding:
Older content refreshed within 30 days showed substantially higher health and impressions than the comparison group.

Methodology question:
Does the comparison/validation design support interpreting the difference as an effect of refreshing, or could pre-existing differences between refreshed and unrefreshed pages explain part of the observed gap?

Why this matters:
The paper is observational and explicitly cautions that correlations do not establish causation. The paper also recommends comparing refreshed pages with comparable untouched pages, which points toward a stronger comparison design for future validation.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

BEFORE: Week-5 client-grouped evaluation
AFTER: Week-6 time-aware evaluation

# Before: Week-5 client-holdout evaluation

In Week 5, I evaluated the models using a client-grouped holdout.
Records from the same client were kept within either the training or
testing partition.

This was a stronger validation design than a simple row-level random
split because it reduced the possibility that client-specific patterns
would appear in both partitions.

However, this design does not fully reproduce the temporal nature of the
prediction problem. A model intended to support future content decisions
should also be evaluated on observations from a later period than the
data used for training.

In [131]:
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
import matplotlib.pyplot as plt

In [132]:
from pathlib import Path
import pandas as pd

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find the repo root containing data/raw/content_refresh_anonymized.csv")

repo_root = find_repo_root()
data_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"

df = pd.read_csv(data_path)
print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [133]:
#Missing Values
numeric_columns = [
    "word_count",
    "char_count",
    "avg_position",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ai_sessions_90d",
    "days_since_last_update"
]

for col in numeric_columns:
    df[col] = df[col].fillna(df[col].median())

In [134]:

# Feature Engineering
# Repository log features

df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

# Repository target

df["is_declining_label"] = (
    df["trend_direction"]
      .str.lower()
      .eq("down")
      .astype(int)
)

In [135]:

# Repository Feature Lists

MODEL_NUMERIC_FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

MODEL_CATEGORICAL_FEATURES = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

TARGET = "is_declining_label"

print("Numeric Features:", len(MODEL_NUMERIC_FEATURES))
print("Categorical Features:", len(MODEL_CATEGORICAL_FEATURES))

Numeric Features: 18
Categorical Features: 8


In [136]:

# Repository Baseline Refresh Score

def percentile_rank(series):
    """Convert values to percentile ranks between 0 and 1."""
    return series.rank(method="average", pct=True)

def normalize(series):
    """Normalize values between 0 and 1."""
    minimum = series.min()
    maximum = series.max()

    if maximum == minimum:
        return pd.Series(np.zeros(len(series)), index=series.index)

    return (series - minimum) / (maximum - minimum)


# Visibility score
df["visibility_score"] = percentile_rank(
    np.log1p(df["impressions_90d"])
)

# Freshness score
df["freshness_risk_score"] = percentile_rank(
    df["days_since_last_update"]
)

# Position opportunity
df["position_opportunity_score"] = (
    (1 - normalize(df["avg_position"].clip(lower=1, upper=50)))
    * df["visibility_score"]
    * (df["avg_position"] > 0).astype(int)
)

# Content depth opportunity
df["depth_gap_score"] = (
    1 - percentile_rank(df["word_count"])
) * df["visibility_score"]

# Final baseline score
df["baseline_refresh_score"] = (
      0.40 * df["visibility_score"]
    + 0.30 * df["freshness_risk_score"]
    + 0.25 * df["position_opportunity_score"]
    + 0.05 * df["depth_gap_score"]
).clip(0, 1)

df["baseline_refresh_score"].describe()

count    30000.000000
mean         0.448383
std          0.214156
min          0.007958
25%          0.282475
50%          0.443107
75%          0.622012
max          0.938382
Name: baseline_refresh_score, dtype: float64

In [137]:

# One-Hot Encode Repository Features

model_df = df[
    MODEL_NUMERIC_FEATURES +
    MODEL_CATEGORICAL_FEATURES +
    [
        "client_id",
        TARGET,
        "baseline_refresh_score"
    ]
].copy()

model_df = pd.get_dummies(
    model_df,
    columns=MODEL_CATEGORICAL_FEATURES,
    drop_first=True
)

print(model_df.shape)
model_df.head()

(30000, 44)


,search_volume,competition,cpc,word_count,char_count,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d,days_with_impressions,...,word_count_tier_2000-3500,word_count_tier_3500+,word_count_tier_<1000,impression_tier_good,impression_tier_low,impression_tier_moderate,position_tier_page_1,position_tier_page_3_5,position_tier_striking,position_tier_top_3
0,10.0,0.67,2.05,3221.0,20457.0,8.243808,3.401197,2.890372,0.0,88,...,True,False,False,True,False,False,False,False,True,False
1,90.0,0.01,0.05,2481.0,15562.0,9.636980,2.079442,2.302585,0.0,88,...,True,False,False,True,False,False,False,True,False,False
2,0.0,0.00,0.00,3515.0,23643.0,9.440023,2.484907,2.484907,0.0,88,...,False,True,False,True,False,False,False,True,False,False
3,10.0,0.00,0.00,2877.0,19116.0,9.371779,4.077537,4.369448,0.0,88,...,False,False,False,True,False,False,True,False,False,False
4,0.0,0.00,0.00,2803.0,17469.0,9.859588,3.218876,4.983607,0.0,88,...,True,False,False,True,False,False,False,True,False,False


In [138]:

# Create X and y

X_train = train_df.drop(
    columns=[
        TARGET,
        "client_id",
        "baseline_refresh_score"
    ]
)

y_train = train_df[TARGET]

X_test = test_df.drop(
    columns=[
        TARGET,
        "client_id",
        "baseline_refresh_score"
    ]
)

y_test = test_df[TARGET]

baseline_scores = test_df["baseline_refresh_score"]

print(X_train.shape)
print(X_test.shape)

(23837, 41)
(6163, 41)



2. My model under an honest split
Before: Week-5 client-holdout evaluation

In Week 5, I evaluated the models using a client-grouped holdout.
The results were:
Model	Precision@20	Precision@50	Precision@100
0	Baseline	0.35	0.28	0.35
1	Logistic Regression	0.90	0.78	0.74
2	Decision Tree	0.40	0.50	0.53
3	Random Forest	0.65	0.64	0.63

In [139]:
[c for c in df.columns if "date" in c.lower() or "month" in c.lower() or "time" in c.lower()]
df.shape
df[["client_id", "content_id"]].nunique()

client_id        32
content_id    30000
dtype: int64

In [140]:
# 1. Check for missing values
print("Missing values:", X_train_fold.isna().sum().sum())

# 2. Check for infinite values
print(
    "Infinite values:",
    np.isinf(X_train_fold.select_dtypes(include=np.number)).sum().sum()
)

# 3. Check the data types
print(X_train_fold.dtypes.value_counts())

Missing values: 6238
Infinite values: 0
bool       23
float64    14
int64       4
Name: count, dtype: int64


In [141]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(
        model_df,
        model_df[TARGET],
        groups=model_df["client_id"]
    ),
    start=1
):
    print(
        f"Fold {fold}: "
        f"Train = {len(train_idx)}, "
        f"Test = {len(test_idx)}"
    )

Fold 1: Train = 22992, Test = 7008
Fold 2: Train = 24269, Test = 5731
Fold 3: Train = 24247, Test = 5753
Fold 4: Train = 24245, Test = 5755
Fold 5: Train = 24247, Test = 5753


In [142]:
# ============================================================
# 5-FOLD CLIENT-GROUPED VALIDATION
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


# ------------------------------------------------------------
# 1. Precision@K
# ------------------------------------------------------------

def precision_at_k(y_true, scores, k):
    ranked = np.argsort(np.asarray(scores))[::-1][:k]
    return np.asarray(y_true)[ranked].mean()


# ------------------------------------------------------------
# 2. Define the actual model features
# ------------------------------------------------------------

TARGET = "is_declining_label"

DROP_COLUMNS = [
    TARGET,
    "client_id",
    "baseline_refresh_score"
]

FEATURES = [
    col for col in model_df.columns
    if col not in DROP_COLUMNS
]

print("Number of model features:", len(FEATURES))
print("Features:")
print(FEATURES)


# ------------------------------------------------------------
# 3. Create 5 client-grouped folds
# ------------------------------------------------------------

gkf = GroupKFold(n_splits=5)

fold_results = []


# ------------------------------------------------------------
# 4. Run the experiment
# ------------------------------------------------------------

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(
        model_df,
        model_df[TARGET],
        groups=model_df["client_id"]
    ),
    start=1
):

    print("\n" + "=" * 60)
    print(f"FOLD {fold}")
    print("=" * 60)

    # --------------------------------------------------------
    # Create train/test data
    # --------------------------------------------------------

    train_fold = model_df.iloc[train_idx].copy()
    test_fold = model_df.iloc[test_idx].copy()

    print("Train rows:", len(train_fold))
    print("Test rows :", len(test_fold))

    # --------------------------------------------------------
    # Create X and y
    # --------------------------------------------------------

    X_train_fold = train_fold[FEATURES].copy()
    y_train_fold = train_fold[TARGET].copy()

    X_test_fold = test_fold[FEATURES].copy()
    y_test_fold = test_fold[TARGET].copy()

    # --------------------------------------------------------
    # Baseline
    # --------------------------------------------------------

    baseline_scores_fold = test_fold[
        "baseline_refresh_score"
    ].copy()


    # --------------------------------------------------------
    # Imputation
    #
    # IMPORTANT:
    # The median is learned ONLY from this fold's training data.
    # --------------------------------------------------------

    imputer = SimpleImputer(strategy="median")

    X_train_fold_imputed = imputer.fit_transform(
        X_train_fold
    )

    X_test_fold_imputed = imputer.transform(
        X_test_fold
    )


    # --------------------------------------------------------
    # Logistic Regression
    # --------------------------------------------------------

    lr_fold = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    lr_fold.fit(
        X_train_fold_imputed,
        y_train_fold
    )

    lr_prob_fold = lr_fold.predict_proba(
        X_test_fold_imputed
    )[:, 1]


    # --------------------------------------------------------
    # Decision Tree
    # --------------------------------------------------------

    dt_fold = DecisionTreeClassifier(
        max_depth=6,
        random_state=42
    )

    dt_fold.fit(
        X_train_fold_imputed,
        y_train_fold
    )

    dt_prob_fold = dt_fold.predict_proba(
        X_test_fold_imputed
    )[:, 1]


    # --------------------------------------------------------
    # Random Forest
    # --------------------------------------------------------

    rf_fold = RandomForestClassifier(
        n_estimators=500,
        max_depth=12,
        min_samples_leaf=3,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1
    )

    rf_fold.fit(
        X_train_fold_imputed,
        y_train_fold
    )

    rf_prob_fold = rf_fold.predict_proba(
        X_test_fold_imputed
    )[:, 1]


    # --------------------------------------------------------
    # Evaluate models
    # --------------------------------------------------------

    model_scores = {
        "Baseline": baseline_scores_fold,
        "Logistic Regression": lr_prob_fold,
        "Decision Tree": dt_prob_fold,
        "Random Forest": rf_prob_fold
    }


    for model_name, scores in model_scores.items():

        fold_results.append({

            "Fold": fold,

            "Model": model_name,

            "Precision@20": precision_at_k(
                y_test_fold,
                scores,
                20
            ),

            "Precision@50": precision_at_k(
                y_test_fold,
                scores,
                50
            ),

            "Precision@100": precision_at_k(
                y_test_fold,
                scores,
                100
            )
        })


# ------------------------------------------------------------
# 5. Convert results to DataFrame
# ------------------------------------------------------------

fold_results_df = pd.DataFrame(
    fold_results
)

print("\nFold-level results:")
display(fold_results_df)

Number of model features: 41
Features:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'competition_level_LOW', 'competition_level_MEDIUM', 'content_type_feedly article', 'content_type_keyword article', 'main_intent_informational', 'main_intent_navigational', 'main_intent_transactional', 'age_tier_31-90', 'age_tier_365+', 'age_tier_91-180', 'freshness_tier_181+', 'freshness_tier_31-90', 'freshness_tier_91-180', 'word_count_tier_2000-3500', 'word_count_tier_3500+', 'word_count_tier_<1000', 'impression_tier_good', 'impression_tier_low', 'impression_tier_moderate', 'position_tier_page_1', 'position_tier_page_3_5', 'position_tier_striking', 'position_tier_top_3']

FOLD 1
Train rows: 22992
Test rows : 7008


c:\Users\Teninlanimi\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



FOLD 2
Train rows: 24269
Test rows : 5731


c:\Users\Teninlanimi\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



FOLD 3
Train rows: 24247
Test rows : 5753


c:\Users\Teninlanimi\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


... (2 outputs left out)


FOLD 5
Train rows: 24247
Test rows : 5753


c:\Users\Teninlanimi\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Fold-level results:


,Fold,Model,Precision@20,Precision@50,Precision@100
0,1,Baseline,0.25,0.34,0.37
1,1,Logistic Regression,0.90,0.76,0.76
2,1,Decision Tree,0.55,0.60,0.62
3,1,Random Forest,0.90,0.86,0.83
4,2,Baseline,0.40,0.44,0.39
5,2,Logistic Regression,0.85,0.82,0.79
6,2,Decision Tree,0.95,0.88,0.84
7,2,Random Forest,0.85,0.88,0.87
8,3,Baseline,0.30,0.30,0.35
9,3,Logistic Regression,0.80,0.62,0.58


In [143]:
# ============================================================
# SUMMARY ACROSS THE 5 CLIENT FOLDS
# ============================================================

cv_summary = (
    fold_results_df
    .groupby("Model")
    .agg(
        P20_mean=("Precision@20", "mean"),
        P20_std=("Precision@20", "std"),

        P50_mean=("Precision@50", "mean"),
        P50_std=("Precision@50", "std"),

        P100_mean=("Precision@100", "mean"),
        P100_std=("Precision@100", "std")
    )
    .reset_index()
)

cv_summary

,Model,P20_mean,P20_std,P50_mean,P50_std,P100_mean,P100_std
0,Baseline,0.43,0.178885,0.472,0.167093,0.468,0.138636
1,Decision Tree,0.47,0.351070,0.508,0.345717,0.512,0.327597
2,Logistic Regression,0.83,0.044721,0.764,0.090995,0.742,0.115629
3,Random Forest,0.76,0.171026,0.708,0.164073,0.712,0.144637


In [144]:

# ============================================================
# SUMMARY ACROSS THE 5 CLIENT FOLDS
# ============================================================

cv_summary = (
    fold_results_df
    .groupby("Model")
    .agg(
        P20_mean=("Precision@20", "mean"),
        P20_std=("Precision@20", "std"),

        P50_mean=("Precision@50", "mean"),
        P50_std=("Precision@50", "std"),

        P100_mean=("Precision@100", "mean"),
        P100_std=("Precision@100", "std")
    )
    .reset_index()
)

cv_summary

,Model,P20_mean,P20_std,P50_mean,P50_std,P100_mean,P100_std
0,Baseline,0.43,0.178885,0.472,0.167093,0.468,0.138636
1,Decision Tree,0.47,0.351070,0.508,0.345717,0.512,0.327597
2,Logistic Regression,0.83,0.044721,0.764,0.090995,0.742,0.115629
3,Random Forest,0.76,0.171026,0.708,0.164073,0.712,0.144637


 Before vs After Validation

The Week-5 evaluation used one client-grouped holdout, while this audit evaluates the same modeling approach across five client-grouped folds. This provides a broader view of how performance varies across held-out client groups.

| Model | Week-5 P@20 | Week-6 Mean P@20 | Week-6 P@20 SD |
|---|---:|---:|---:|
| Baseline | 0.35 | 0.43 | 0.179 |
| Logistic Regression | 0.90 | 0.83 | 0.045 |
| Decision Tree | 0.40 | 0.47 | 0.351 |
| Random Forest | 0.65 | 0.76 | 0.171 |

The Week-5 single-holdout results were higher for Logistic Regression (P@20 = 0.90) than its five-fold mean (0.83). Across the five folds, however, Logistic Regression showed relatively low variation (SD = 0.045), while the Decision Tree showed substantially higher variation (SD = 0.351). These results provide decision-support evidence that Logistic Regression was comparatively stable across the held-out client groups evaluated here.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*



I reviewed the features used by the Week-5 model to identify variables that could directly contain the target, encode the outcome, or use information that would not be available at prediction time.

In [145]:
# Features actually used by the model

TARGET = "is_declining_label"

DROP_COLUMNS = [
    TARGET,
    "client_id",
    "baseline_refresh_score"
]

MODEL_FEATURES = [
    col for col in model_df.columns
    if col not in DROP_COLUMNS
]

print("Number of model features:", len(MODEL_FEATURES))
print(MODEL_FEATURES)

Number of model features: 41
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'competition_level_LOW', 'competition_level_MEDIUM', 'content_type_feedly article', 'content_type_keyword article', 'main_intent_informational', 'main_intent_navigational', 'main_intent_transactional', 'age_tier_31-90', 'age_tier_365+', 'age_tier_91-180', 'freshness_tier_181+', 'freshness_tier_31-90', 'freshness_tier_91-180', 'word_count_tier_2000-3500', 'word_count_tier_3500+', 'word_count_tier_<1000', 'impression_tier_good', 'impression_tier_low', 'impression_tier_moderate', 'position_tier_page_1', 'position_tier_page_3_5', 'position_tier_striking', 'position_tier_top_3']


In [146]:
# Look for feature names that directly reference the target
target_related_features = [
    col for col in MODEL_FEATURES
    if "declin" in col.lower()
    or "label" in col.lower()
    or "target" in col.lower()
]

print("Potential target-related features:")
print(target_related_features)

Potential target-related features:
[]


In [147]:
#check of target is accidentaly in the feature
print("Target included in features?", TARGET in MODEL_FEATURES)

Target included in features? False


### Initial target leakage check

The model used 41 features after excluding `is_declining_label`, `client_id`, and `baseline_refresh_score`. No model feature had a name containing `declin`, `label`, or `target`, and the target was confirmed to be absent from the feature matrix.

This screen did not identify obvious direct target leakage. However, feature names alone cannot establish that the features are leakage-free, so the next step is to review the feature definitions and their timing relative to the prediction point.

In [148]:
# Check for date/time columns in model_df

date_columns = [
    col for col in model_df.columns
    if "date" in col.lower()
    or "month" in col.lower()
    or "time" in col.lower()
]

print("Date/time columns:")
print(date_columns)
print(model_df.columns.tolist())


Date/time columns:
['days_since_last_update']
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'client_id', 'is_declining_label', 'baseline_refresh_score', 'competition_level_LOW', 'competition_level_MEDIUM', 'content_type_feedly article', 'content_type_keyword article', 'main_intent_informational', 'main_intent_navigational', 'main_intent_transactional', 'age_tier_31-90', 'age_tier_365+', 'age_tier_91-180', 'freshness_tier_181+', 'freshness_tier_31-90', 'freshness_tier_91-180', 'word_count_tier_2000-3500', 'word_count_tier_3500+', 'word_count_tier_<1000', 'impression_tier_good', 'impression_tier_low', 'impression_tier_moderate', 'position_tier_page_1', 'position_tier_page_3_5', 'position_tier_striking', 'position_tier_

In [149]:
numeric_model_features = [
    col for col in MODEL_FEATURES
    if pd.api.types.is_numeric_dtype(model_df[col])
]

feature_target_summary = (
    model_df
    .groupby(TARGET)[numeric_model_features]
    .mean()
    .T
)

feature_target_summary["difference"] = (
    feature_target_summary[1]
    - feature_target_summary[0]
)

feature_target_summary.sort_values(
    "difference",
    key=abs,
    ascending=False
).head(15)

is_declining_label,0,1,difference
char_count,19614.040472,20819.871910,1205.831438
word_count,2933.348522,3145.851925,212.503402
search_volume,191.861414,133.376490,-58.484925
content_age_days,279.829451,236.178637,-43.650814
days_with_impressions,54.694715,67.164248,12.469533
days_since_last_update,42.372543,49.245788,6.873244
log_impressions_90d,5.669569,6.627235,0.957666
avg_position,16.823060,15.936305,-0.886755
days_with_sessions,13.568132,12.697762,-0.870371
ctr,0.731611,0.324138,-0.407473


### Feature timing and construction review

The initial feature screen found no feature that directly referenced the target, and the target was excluded from the model matrix. Several features show different average values between declining and non-declining observations, but this alone does not establish leakage.

I therefore reviewed the feature definitions for whether they could contain information from after the prediction point. In particular, the historical performance features (`*_90d`, `days_with_*`, `ctr`, `avg_position`, and engagement measures) require a timing check because their validity depends on the period used to calculate them.

In [150]:
print("trend_direction in model_df:",
      "trend_direction" in model_df.columns)

print("trend_direction in df:",
      "trend_direction" in df.columns)
[c for c in df.columns if "trend" in c.lower()]

trend_direction in model_df: False
trend_direction in df: True


['trend_direction', 'trend_pct']

In [151]:
print(df[["trend_direction", "is_declining_label"]].head(20))
print(df["trend_direction"].value_counts(dropna=False))
print(df.columns.tolist())

   trend_direction  is_declining_label
0             down                   1
1             down                   1
2             down                   1
3           stable                   0
4             down                   1
5             down                   1
6             down                   1
7           stable                   0
8             down                   1
9             down                   1
10          stable                   0
11             new                   0
12              up                   0
13          stable                   0
14            down                   1
15              up                   0
16            down                   1
17            down                   1
18            down                   1
19            down                   1
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64
['content_id', 'client_id', 'search_volume', 'competition',

In [152]:
print(
    df[
        [
            "impressions_last_30d",
            "impressions_prev_30d",
            "clicks_last_30d",
            "clicks_prev_30d",
            "sessions_last_30d",
            "sessions_prev_30d",
            "trend_pct",
            "trend_direction"
        ]
    ].head(20)
)


print(df["trend_pct"].describe())

print(
    df[
        [
            "impressions_last_30d",
            "impressions_prev_30d",
            "clicks_last_30d",
            "clicks_prev_30d",
            "sessions_last_30d",
            "sessions_prev_30d",
            "trend_pct"
        ]
    ].corr()["trend_pct"].sort_values()
)

    impressions_last_30d  impressions_prev_30d  clicks_last_30d  \
0                    578                   987                2   
1                   2501                  5915                2   
2                   2382                  6089                1   
3                   3626                  4206               22   
4                   4211                  6452               10   
5                    617                  1009                0   
6                      1                    13                0   
7                    636                   632                1   
8                   5696                 13828                9   
9                    252                   356                0   
10                  7665                  6441              112   
11                     1                     0                0   
12                  5127                  1357               92   
13                    85                    77                

In [153]:
pd.crosstab(
    df["trend_direction"],
    df["is_declining_label"]
)
df.groupby("trend_direction")["trend_pct"].agg(
    ["count", "min", "mean", "median", "max"]
)


,count,min,mean,median,max
trend_direction,,,,,
down,16262,-100.0,-58.113830,-55.60,-20.0
flat,0,NaN,NaN,NaN,NaN
new,0,NaN,NaN,NaN,NaN
stable,5962,-20.0,-3.185944,-3.80,20.0
up,4388,20.0,190.673997,62.55,44900.0


 Target provenance check

The target `is_declining_label` is derived directly from `trend_direction`: observations with `trend_direction == "down"` are encoded as 1, while other categories are encoded as 0.

`trend_direction` and `trend_pct` are present in the source dataframe but are not included in the final `model_df` feature matrix. Therefore, the model does not directly receive either the target-construction field or `trend_pct`.

The original transformation that produced `trend_direction` was not found in the notebook. Because the source data contains both recent-versus-previous-period performance fields and trend variables, the timing relationship between the label and historical performance features requires further review before claiming that all features are leakage-free.

3.2 Target construction and feature leakage review

The target `is_declining_label` is created by encoding observations with `trend_direction == "down"` as 1 and all other trend categories as 0.

The source data also contains `trend_pct`, which is associated with the trend categories. In the observed data, the `down` category has trend percentages from -100% to -20%, while the `stable` category ranges from -20% to +20% and the `up` category begins at +20%.

Neither `trend_direction` nor `trend_pct` is included in the final model feature matrix. The recent-period variables used alongside the trend calculation (`impressions_last_30d`, `impressions_prev_30d`, `clicks_last_30d`, `clicks_prev_30d`, `sessions_last_30d`, and `sessions_prev_30d`) are also not included in the model features.

The model instead uses aggregated historical features such as `log_impressions_90d`, `log_clicks_90d`, `log_sessions_90d`, `days_with_impressions`, and `days_with_sessions`.

I did not identify direct target leakage in the final feature matrix. However, the original transformation used to create `trend_direction` could not be recovered from the available notebook, so the exact timing relationship between the label and the historical feature windows could not be independently verified. This remains a data-lineage limitation.

 3.3 Feature-level leakage assessment

The following review focuses on whether features appear to contain the target directly or represent information that would only be available after the prediction point.

In [154]:
leakage_review = pd.DataFrame({
    "Feature": [
        "log_impressions_90d",
        "log_clicks_90d",
        "log_sessions_90d",
        "log_ai_sessions_90d",
        "days_with_impressions",
        "days_with_sessions",
        "content_age_days",
        "days_since_last_update",
        "ctr",
        "avg_position",
        "engagement_rate",
        "scroll_rate",
        "ai_traffic_pct"
    ],
    "Assessment": [
        "Historical aggregate; timing should be verified",
        "Historical aggregate; timing should be verified",
        "Historical aggregate; timing should be verified",
        "Historical aggregate; timing should be verified",
        "Historical count; timing should be verified",
        "Historical count; timing should be verified",
        "Content metadata; no direct target reference",
        "Content freshness metadata; no direct target reference",
        "Historical performance metric; timing should be verified",
        "Historical performance metric; timing should be verified",
        "Historical performance metric; timing should be verified",
        "Historical performance metric; timing should be verified",
        "Historical performance metric; timing should be verified"
    ]
})

leakage_review

,Feature,Assessment
0,log_impressions_90d,Historical aggregate; timing should be verified
1,log_clicks_90d,Historical aggregate; timing should be verified
2,log_sessions_90d,Historical aggregate; timing should be verified
3,log_ai_sessions_90d,Historical aggregate; timing should be verified
4,days_with_impressions,Historical count; timing should be verified
5,days_with_sessions,Historical count; timing should be verified
6,content_age_days,Content metadata; no direct target reference
7,days_since_last_update,Content freshness metadata; no direct target r...
8,ctr,Historical performance metric; timing should b...
9,avg_position,Historical performance metric; timing should b...


Leakage audit conclusion

No direct target leakage was identified in the final model feature matrix. The target construction fields `trend_direction` and `trend_pct`, as well as the recent-period fields used alongside the trend calculation, were excluded from the model features.

However, the original transformation that created `trend_direction` could not be recovered from the available notebook. Therefore, I could not fully verify whether the historical performance windows used by features such as `log_impressions_90d`, `ctr`, and `avg_position` were strictly prior to the label-generation period.

I therefore treat the model as having **no identified direct leakage**, while reporting feature/label timing as an unresolved data-lineage limitation. This limits how strongly the validation results should be interpreted.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*



### Original claim

> Logistic Regression is the best model for identifying declining content and can accurately prioritize pages for refresh.

### Rewritten claim

> In the client-holdout validation, Logistic Regression showed the strongest observed ranking performance among the tested models, with mean Precision@20 of 0.83 across five folds. This suggests it can provide useful decision-support for prioritizing potentially declining content for review, but the result should be treated as directional rather than as evidence that the model will reliably identify every declining page.

### Why I changed it

The original claim was based primarily on the Week-5 evaluation and was stronger than the evidence supported. The client-holdout validation produced lower and more variable performance across folds, showing that model performance depends on which clients are held out.

The revised claim focuses on observed ranking performance rather than general predictive accuracy. It positions the model as decision-support for prioritization rather than as a definitive predictor of content decline.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.